# 82514 · Sesión S24 — Una política sin el modelo y el simulador

**Bloque 8** · lunes 14 de diciembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** El péndulo sobre un carro de Gymnasium, que integra la física a 50 Hz. Una política de cuatro pesos que empuja a un lado o a otro se encuentra sin el modelo, probando: al azar, con búsqueda aleatoria y con la entropía cruzada. Después, la brecha: la misma política con otro palo y con un retardo entre medir y actuar; y la aleatorización de dominio, que la arregla dentro de lo que se varió y no fuera. En casa, PPO de verdad con Stable-Baselines3.

**Se apoya en:** Sutton y Barto (2018); Schulman et al. (2017), PPO; Tobin et al. (2017), la aleatorización de dominio; documentación de Gymnasium y de Stable-Baselines3.

**Cómo usarlo en clase.** Sigue el guion de la sesión S24 en las diapositivas del bloque 8. Ejecuta la primera celda una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos.
import os, sys, importlib, importlib.metadata as md, subprocess
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')    # pygame (el dibujo del péndulo) sin pantalla y sin avisos

FIJAS = {'gymnasium': '1.4.0', 'scikit-learn': '1.6.1', 'stable-baselines3': '2.9.0'}   # las del bloque 8
COLAB = {'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'torch': '2.11.0'}  # se comprueban, no se instalan
USA = ['gymnasium']             # las del bloque que usa este cuaderno
EN_COLAB = 'google.colab' in sys.modules

def version(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

for paquete in USA:
    if paquete == 'stable-baselines3' and version('torch') is None:
        print('AVISO: no hay torch en este entorno: la sección de PPO no se ejecuta (Colab sí lo trae).')
        continue
    if version(paquete) != FIJAS[paquete]:
        print('Instalando', paquete, FIJAS[paquete], '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{paquete}=={FIJAS[paquete]}'], check=False)
importlib.invalidate_caches()

import math, random, time
import numpy as np, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
for p in ['numpy', 'scipy', 'matplotlib'] + USA:
    w = (version(p) or 'no instalada').split('+')[0]
    v = {**COLAB, **FIJAS}[p]
    print(f'{p:18s} {w:12s}', '' if w == v else f'(probado con {v})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

plt.rcParams['figure.figsize'] = (11, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def pc(x):
    """Una fracción como porcentaje entero."""
    return f'{100 * x:.0f} %'


import gymnasium as gym

> **Aviso para Colab.** Ábrelo al empezar la sesión y ejecuta la primera celda: comprueba gymnasium 1.4.0 (Colab ya la trae) y tarda unos segundos.
>
> **En clase (siete minutos):** la sección 1 sin detenerte, y el taller, de la 2 a la 4. Las cifras tienen que salir como en las diapositivas. Si al volver a él Colab dice que se ha desconectado (pasa tras un rato largo sin usarlo), conéctalo de nuevo (arriba a la derecha) y ejecuta otra vez la primera celda y la de la sección 1: unos segundos.
>
> **En casa:** la sección 5 (PPO, unos 20 s; si hay torch, instala antes stable-baselines3 2.9.0, de 0,2 MB) y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.

## 1. El péndulo sobre un carro: un simulador con física (diapositiva 21)

*En clase, ejecuta estas celdas sin detenerte.*

El almacén de S21 solo integraba velocidades. Este simulador integra **F = m·a** a 50 Hz: un carro de 1 kg empujado con 10 N a izquierda o derecha, y sobre él un palo de 0,1 kg que hay que mantener de pie. Gymnasium no es el simulador: es la forma común de hablar con cualquiera de ellos, con dos llamadas.

- `obs, info = env.reset(seed=...)` empieza un episodio y da la primera observación.
- `obs, r, terminated, truncated, info = env.step(a)` aplica la acción y da la observación siguiente, la recompensa (+1 por cada paso con el palo de pie), y si el episodio ha acabado porque el palo cae (`terminated`) o por tiempo (`truncated`).

La observación son cuatro números: la posición y la velocidad del carro, el ángulo del palo y su velocidad angular. Aquí el episodio se corta en 300 pasos (6 s): el retorno va de 0 a 300.

El código de esta sección y de las siguientes es el mismo que dio las cifras de las diapositivas.

In [ ]:
# El péndulo, la política lineal, la entropía cruzada y la aleatorización (el mismo código que las diapositivas)
MAX_PASOS = 300                  # tope por episodio (CartPole-v1 trunca de serie en 500)


def ajustar_fisica(env, masspole=0.1, length=0.5, force_mag=10.0):
    """Cambia la física del péndulo (masa del palo en kg, media longitud en m, fuerza en N). Hay dos cantidades
    derivadas que el simulador calcula al crearse y que hay que recalcular a mano."""
    u = env.unwrapped
    u.masspole, u.length, u.force_mag = masspole, length, force_mag
    u.total_mass = u.masscart + u.masspole
    u.polemass_length = u.masspole * u.length


def episodio(env, w, semilla, max_pasos=MAX_PASOS, retardo=0):
    """Un episodio con la política lineal: empuja a la derecha si w·x > 0 y a la izquierda si no (todo o nada).
    'retardo' son los pasos de 20 ms entre medir y actuar."""
    obs, _ = env.reset(seed=int(semilla))
    memoria = [obs] * (retardo + 1)
    total = 0.0
    for _ in range(max_pasos):
        a = 1 if float(w @ memoria[0]) > 0.0 else 0     # actúa con la observación retrasada
        obs, r, term, trunc, _ = env.step(a)
        memoria = memoria[1:] + [obs]
        total += r
        if term or trunc:
            break
    return total


def evaluar(env, w, n=20, semilla_base=9000, **kw):
    """El retorno medio de n episodios (de 0 a 300: los pasos que el palo aguanta de pie)."""
    return float(np.mean([episodio(env, w, semilla_base + i, **kw) for i in range(n)]))


def busqueda_aleatoria(env, rng, n_candidatos=360, n_ep=3, **kw):
    """Probar pesos al azar y quedarse con el mejor."""
    mejor_w, mejor_s, historia = None, -np.inf, []
    for k in range(n_candidatos):
        w = rng.normal(size=4)
        s = np.mean([episodio(env, w, 7 * k + i, **kw) for i in range(n_ep)])
        if s > mejor_s:
            mejor_w, mejor_s = w, s
        historia.append(mejor_s)
    return mejor_w, np.array(historia)


def entropia_cruzada(env, rng, evaluador=None, n_iter=15, n_cand=30, n_elite=6, n_ep=3):
    """La entropía cruzada (CEM): probar 30 pesos, quedarse con los 6 mejores y sacar de ellos la siguiente tanda."""
    if evaluador is None:
        evaluador = lambda w, k: np.mean([episodio(env, w, 7 * k + i) for i in range(n_ep)])
    mu, sigma = np.zeros(4), np.ones(4)
    historia = []
    for it in range(n_iter):
        candidatos = rng.normal(mu, sigma, size=(n_cand, 4))
        puntos = np.array([evaluador(w, it * n_cand + k) for k, w in enumerate(candidatos)])
        elite = candidatos[np.argsort(puntos)[-n_elite:]]
        mu, sigma = elite.mean(axis=0), elite.std(axis=0) + 0.02      # el + 0,02 evita que se cierre del todo
        historia.append((puntos.mean(), puntos.max()))
    return mu, np.array(historia)


MASAS, LONGITUDES, RETARDOS = np.array([0.1, 0.3, 0.6, 1.0]), np.array([0.5, 0.8, 1.1, 1.5]), [0, 2, 3]


def rejilla_fisica(env, w, masas=MASAS, longitudes=LONGITUDES, retardo=0, n=6):
    """El retorno medio de la política w en cada mundo de la rejilla de masas y longitudes, con un retardo."""
    M = np.zeros((len(masas), len(longitudes)))
    for i, mp in enumerate(masas):
        for j, L in enumerate(longitudes):
            ajustar_fisica(env, masspole=mp, length=L)
            M[i, j] = evaluar(env, w, n=n, semilla_base=5000, retardo=retardo)
    ajustar_fisica(env)
    return M


def evaluador_aleatorizado(env, w, k, n_ep=3):
    """La aleatorización de dominio: cada episodio, un péndulo distinto (masa de 0,05 a 1,2 kg, media longitud de 0,3
    a 1,5 m y un retardo de 0 a 4 pasos, de 0 a 80 ms)."""
    r = np.random.default_rng(k)
    puntos = []
    for i in range(n_ep):
        ajustar_fisica(env, masspole=r.uniform(0.05, 1.2), length=r.uniform(0.3, 1.5))
        puntos.append(episodio(env, w, 7 * k + i, retardo=int(r.integers(0, 5))))
    ajustar_fisica(env)
    return float(np.mean(puntos))


env = gym.make('CartPole-v1')
u = env.unwrapped
print(f'carro {coma(u.masscart, 1)} kg; palo {coma(u.masspole, 1)} kg con media longitud {coma(u.length, 1)} m; '
      f'fuerza {coma(u.force_mag, 0)} N; {1 / u.tau:.0f} Hz')
obs, info = env.reset(seed=0)
print('primera observación (x, ẋ, θ, θ̇):', '; '.join(coma(v, 3) for v in obs))
obs, r, terminated, truncated, info = env.step(1)
print('tras empujar a la derecha:', '; '.join(coma(v, 3) for v in obs), f'; r = {r:g}; terminated = {terminated}')

try:
    vista = gym.make('CartPole-v1', render_mode='rgb_array')     # un fotograma, dibujado por el propio Gymnasium
    vista.reset(seed=0)
    for _ in range(8):
        vista.step(1)
    foto = vista.render()
    vista.close()
except gym.error.DependencyNotInstalled:                         # el dibujo usa pygame: Colab lo trae
    foto = None
    print('(sin pygame no se dibuja el fotograma; Colab lo trae)')
if foto is not None:
    plt.figure(figsize=(5, 3.4)); plt.imshow(foto); plt.axis('off')
    plt.title('El péndulo sobre un carro, tras ocho empujones a la derecha', fontsize=10); plt.show()

## 2. Taller: una política sin el modelo (diapositivas 19 y 23)

La política es una **realimentación de estado todo o nada**: cuatro pesos w, y empuja a la derecha si w·x > 0 y a la izquierda si no. Nadie escribe el modelo del péndulo: se prueban pesos y se mide el retorno. Tres maneras, de peor a mejor:

- **Al azar:** 200 vectores w, cada uno con cinco episodios. Es la línea base: sin ella, ninguna cifra dice nada.
- **Búsqueda aleatoria:** probar 360 vectores y quedarse con el mejor.
- **Entropía cruzada:** una gaussiana sobre los pesos; en cada vuelta, 30 candidatos, los 6 mejores, y la gaussiana nueva sale de esos 6. Quince vueltas.

Cada candidato se mide con tres episodios: con uno solo, se elegiría al que tuvo suerte (el ejercicio 3).

**Taller.** Ejecuta: tiene que salir 47,0 al azar, 263,9 con la búsqueda aleatoria y 300,0 con la entropía cruzada, con los pesos de la diapositiva.

In [ ]:
def pesos(w):
    return '[' + '; '.join(coma(v, 3) for v in w) + ']'


rng = np.random.default_rng(39)
ajustar_fisica(env)
al_azar = np.array([evaluar(env, rng.normal(size=4), n=5, semilla_base=100 * k) for k in range(200)])
print(f'200 al azar: media {coma(al_azar.mean(), 1)} de {MAX_PASOS}')

rng = np.random.default_rng(39)
t0 = time.perf_counter()
w_ra, hist_ra = busqueda_aleatoria(env, rng)
w_cem, hist_cem = entropia_cruzada(env, rng)
print(f'el mejor de 360: {coma(evaluar(env, w_ra), 1)}')
print(f'entropía cruzada: {coma(evaluar(env, w_cem), 1)}')
print(f'pesos {pesos(w_cem)}')
print(f'(mediana al azar {coma(np.median(al_azar), 1)}; pasa de 100 el {pc(np.mean(al_azar > 100))}; '
      f'las dos búsquedas, en {coma(time.perf_counter() - t0, 1)} s)')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.3))
a1.hist(al_azar, bins=30, color=GRIS, edgecolor='white')
a1.axvline(al_azar.mean(), color=ROJO, lw=2, ls='--', label='media al azar')
a1.axvline(evaluar(env, w_cem), color=IQS_VERDE, lw=2.5, label='entropía cruzada')
a1.set_xlabel('retorno (pasos de pie)'); a1.set_ylabel('políticas'); a1.legend(fontsize=8)
a1.set_title('200 políticas al azar', fontsize=10)
a2.plot(np.arange(1, len(hist_ra) + 1) * 3, hist_ra, color=IQS_AZUL, lw=2, label='búsqueda aleatoria (la mejor)')
a2.plot(np.arange(1, 16) * 90, hist_cem[:, 0], color=IQS_VERDE, lw=2, label='entropía cruzada (media de la vuelta)')
a2.plot(np.arange(1, 16) * 90, hist_cem[:, 1], color=IQS_VERDE, lw=1, ls='--', label='entropía cruzada (la mejor)')
a2.set_xlabel('episodios simulados'); a2.set_ylabel('retorno'); a2.legend(fontsize=7)
a2.set_title('Cómo mejoran', fontsize=10)
for a in (a1, a2):
    a.xaxis.set_major_formatter(COMA); a.yaxis.set_major_formatter(COMA)
plt.tight_layout(); plt.show()

En el péndulo, la búsqueda aleatoria casi basta: con cuatro pesos, uno de cada ocho vectores al azar ya aguanta más de 100 pasos. La diferencia está en la otra curva: con la entropía cruzada sube la **media** de cada vuelta, no solo la mejor. Usa lo que mide para moverse, y eso es lo que escala cuando hay más pesos (el ejercicio 1).

## 3. Taller: la brecha (diapositivas 22 y 24)

La política se ha encontrado en un mundo: palo de 0,1 kg y media longitud de 0,5 m, y sin retardo. Se mide ahora en 16 mundos: cuatro masas del palo (0,1; 0,3; 0,6 y 1,0 kg) por cuatro medias longitudes (0,5; 0,8; 1,1 y 1,5 m), y con un **retardo** entre medir y actuar, en pasos de 20 ms. En S20, 100 ms de retraso eran 5 cm de camino; en S21, un barrido llegaba 120 ms tarde: 60 ms es un retardo del curso.

**Taller.** Ejecuta con RETARDO = 3 (60 ms) y mira la media de los 16 mundos y el peor. Si te sobra tiempo, prueba 0 y 2.

In [ ]:
RETARDO = 3      # pasos de 20 ms; prueba 0 y 2

nominal = rejilla_fisica(env, w_cem, retardo=RETARDO)
print(f'nominal, retardo {20 * RETARDO} ms: media {coma(nominal.mean(), 1)}; peor {coma(nominal.min(), 1)}')
print(f'(en su mundo, sin retardo: {coma(evaluar(env, w_cem), 1)})')


def mapa(ax, M, titulo):
    im = ax.imshow(M, vmin=0, vmax=MAX_PASOS, cmap='RdYlGn', aspect='auto', origin='lower')
    ax.set_xticks(range(len(LONGITUDES)), [coma(x, 1) for x in LONGITUDES])
    ax.set_yticks(range(len(MASAS)), [coma(x, 1) for x in MASAS])
    ax.set_xlabel('media longitud del palo (m)'); ax.set_ylabel('masa del palo (kg)')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f'{M[i, j]:.0f}', ha='center', va='center', fontsize=8)
    ax.set_title(titulo, fontsize=10)
    return im


fig, ax = plt.subplots(figsize=(5, 3.6))
mapa(ax, nominal, f'Nominal, retardo {20 * RETARDO} ms'); plt.show()

## 4. Taller: la aleatorización de dominio (diapositivas 22 y 24)

El remedio no es un simulador perfecto: es **variar en cada episodio** lo que no se sabe. La entropía cruzada es la misma; solo cambia cómo se mide cada candidato: en cada episodio, un palo de 0,05 a 1,2 kg, una media longitud de 0,3 a 1,5 m y un retardo de 0 a 80 ms, al azar.

Se mide con 60 ms de retardo, el de la diapositiva 23, aunque en la sección 3 hayas probado otros.

**Taller.** Ejecuta (unos segundos) y compara con la sección 3.

In [ ]:
RETARDO_TALLER = 3      # 60 ms, el de la diapositiva 23

t0 = time.perf_counter()
w_dr, hist_dr = entropia_cruzada(env, np.random.default_rng(39), evaluador=lambda w, k: evaluador_aleatorizado(env, w, k))
nominal60 = rejilla_fisica(env, w_cem, retardo=RETARDO_TALLER)
aleatorizada = rejilla_fisica(env, w_dr, retardo=RETARDO_TALLER)
print(f'con aleatorización, retardo {20 * RETARDO_TALLER} ms: media {coma(aleatorizada.mean(), 1)}; peor {coma(aleatorizada.min(), 1)}')
print(f'(pesos {pesos(w_dr)}, frente a {pesos(w_cem)}; en el mundo nominal, {coma(evaluar(env, w_dr), 1)}; '
      f'{coma(time.perf_counter() - t0, 1)} s)')

fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.6))
mapa(axs[0], nominal60, f'Nominal, retardo {20 * RETARDO_TALLER} ms')
mapa(axs[1], aleatorizada, f'Con aleatorización, retardo {20 * RETARDO_TALLER} ms')
plt.tight_layout(); plt.show()

Fíjate en los pesos: la política aleatorizada casi no mira la posición del carro y carga más el ángulo. Y en el mundo nominal no pierde nada: las dos dan 300. **Solo promete dentro de lo que se varió**: el ejercicio 2 lo comprueba fuera.

## 5. En casa: PPO con Stable-Baselines3 (diapositiva 20)

Stable-Baselines3 es una biblioteca abierta de aprendizaje por refuerzo con PyTorch. Su PPO usa una red de 64 × 64 (miles de pesos, no cuatro), el gradiente de política con la ventaja y el recorte de 0,2 de la diapositiva 20. Con 20 000 pasos del péndulo tarda unos 20 s en la CPU de Colab y llega a unos 500 (aquí el episodio no se corta en 300).

Dos avisos de ingeniero. **PPO no da la misma cifra en todas las máquinas**: con la misma semilla, cambia con el número de hilos de la CPU; por eso se fijan dos hilos y se dan las cifras con «unos». Y **la evaluación se siembra a mano**, episodio a episodio.

Solo se ejecuta si hay torch (Colab lo trae). La celda instala antes stable-baselines3 2.9.0, la versión fijada del bloque (0,2 MB), si no está; torch no se instala nunca.

In [ ]:
PPO = None
if version('torch') is None:
    print('AVISO: no hay torch en este entorno: esta sección no se ejecuta (Colab sí lo trae).')
else:
    if version('stable-baselines3') != FIJAS['stable-baselines3']:
        print('Instalando stable-baselines3', FIJAS['stable-baselines3'], '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f"stable-baselines3=={FIJAS['stable-baselines3']}"], check=False)
        importlib.invalidate_caches()
    for p in ('torch', 'stable-baselines3'):
        w = (version(p) or 'no instalada').split('+')[0]
        v = {**COLAB, **FIJAS}[p]
        print(f'{p:18s} {w:12s}', '' if w == v else f'(probado con {v})')
    try:
        import torch
        from stable_baselines3 import PPO
    except ImportError:
        print('Sin stable-baselines3: esta sección no se ejecuta.')

if PPO is not None:
    torch.set_num_threads(2)                     # con otro número de hilos, otras cifras
    t0 = time.perf_counter()
    modelo = PPO('MlpPolicy', gym.make('CartPole-v1'), seed=0, device='cpu', verbose=0)
    modelo.learn(total_timesteps=20_000)
    print(f'PPO, 20 000 pasos en {coma(time.perf_counter() - t0, 1)} s')
    prueba = gym.make('CartPole-v1')             # trunca en 500

    def retorno_ppo(masspole=0.1, length=0.5, n=10):
        ajustar_fisica(prueba, masspole, length)
        r = []
        for i in range(n):
            o, _ = prueba.reset(seed=500 + i)    # la evaluación, sembrada
            total = 0.0
            for _ in range(500):
                a, _ = modelo.predict(o, deterministic=True)
                o, rr, te, tr, _ = prueba.step(int(a))
                total += rr
                if te or tr:
                    break
            r.append(total)
        ajustar_fisica(prueba)
        return float(np.mean(r))

    print(f'retorno medio en 10 episodios: {coma(retorno_ppo(), 1)} de 500')
    print(f'con el palo el doble de largo (media longitud 1,0 m): {coma(retorno_ppo(0.1, 1.0), 1)}')
    print(f'con 1,0 kg y media longitud 1,5 m: {coma(retorno_ppo(1.0, 1.5), 1)}')

Las dos últimas líneas son la misma red con otro palo, sin retardo: aguanta (unos 500; con un hilo sale 495,9 y 490,2). Con 1,0 kg y media longitud 1,5 m, la política de cuatro pesos se cae aun sin retardo: en la sección 3, con RETARDO = 0, esa casilla da 16. No basta para decir que PPO sea más robusta: son dos mundos, una semilla y sin retardo; para compararlas, habría que medir la red en la rejilla de la sección 3, con 60 ms.

### Ejercicio 1 (en casa): 48 pesos en lugar de 4

Cambia la política lineal por una red diminuta: una capa oculta de 8 neuronas con tangente hiperbólica, 48 pesos. Compara la búsqueda al azar, 150 redes con tres episodios cada una (450 episodios), con la entropía cruzada, que gasta 1350 (15 vueltas de 30 candidatos, con tres episodios cada uno). ¿Qué fracción de las redes al azar pasa de 200? ¿A dónde llega cada método? La celda lo tiene montado.

In [ ]:
# Ejercicio 1: la misma búsqueda, con 48 pesos
H = 8
DIM = 4 * H + H + H


def accion_red(theta, obs):
    W1 = theta[:4 * H].reshape(4, H); b1 = theta[4 * H:5 * H]; w2 = theta[5 * H:]
    return 1 if float(w2 @ np.tanh(obs @ W1 + b1)) > 0 else 0


def episodio_red(theta, semilla, max_pasos=MAX_PASOS):
    obs, _ = env.reset(seed=int(semilla)); total = 0.0
    for _ in range(max_pasos):
        obs, r, term, trunc, _ = env.step(accion_red(theta, obs)); total += r
        if term or trunc:
            break
    return total


ajustar_fisica(env)
rng_e = np.random.default_rng(3)
puntos = np.array([np.mean([episodio_red(w, 100 * k + i) for i in range(3)])
                   for k, w in enumerate(rng_e.normal(size=(150, DIM)))])
print(f'al azar, {DIM} pesos: media {coma(puntos.mean(), 1)}; pasa de 200 el {pc(np.mean(puntos > 200))}; '
      f'la mejor {coma(puntos.max(), 1)}')
print(f'(con 4 pesos, de las 200 de la sección 2, pasa de 200 el {pc(np.mean(al_azar > 200))})')
mu, sigma = np.zeros(DIM), np.ones(DIM)
for it in range(15):
    cand = rng_e.normal(mu, sigma, size=(30, DIM))
    sc = np.array([np.mean([episodio_red(w, 7 * (it * 30 + k) + i) for i in range(3)]) for k, w in enumerate(cand)])
    elite = cand[np.argsort(sc)[-6:]]
    mu, sigma = elite.mean(axis=0), elite.std(axis=0) + 0.02
print(f'entropía cruzada, {DIM} pesos: última vuelta {coma(sc.mean(), 1)}; la final, '
      f'{coma(np.mean([episodio_red(mu, 50000 + i) for i in range(20)]), 1)}')

### Ejercicio 2 (en casa): fuera de lo variado

Mide las dos políticas fuera del rango de la aleatorización: palos de 2, 3 y 4 kg, medias longitudes de 2, 2,5 y 3 m, y retardos de 6 y 8 pasos (120 y 160 ms). ¿Sigue ganando la aleatorizada? ¿Se sostiene? Después, ensancha la aleatorización hasta cubrir ese mundo (palo de 0,05 a 4 kg, media longitud de 0,3 a 3 m y retardo de 0 a 120 ms) y vuelve a medir: fuera, en el mundo nominal y en la rejilla de la sección 3 con 60 ms. ¿Era imposible el mundo de fuera? Di con tus palabras qué promete la aleatorización de dominio y qué no.

In [ ]:
# Ejercicio 2
MASAS_X, LONG_X = np.array([2.0, 3.0, 4.0]), np.array([2.0, 2.5, 3.0])
for d in (6, 8):
    a = rejilla_fisica(env, w_cem, MASAS_X, LONG_X, retardo=d, n=4).mean()
    b = rejilla_fisica(env, w_dr, MASAS_X, LONG_X, retardo=d, n=4).mean()
    print(f'retardo {20 * d} ms: nominal {coma(a, 1)}; aleatorizada {coma(b, 1)}')


def evaluador_ancho(w, k, n_ep=3):
    """La aleatorización ensanchada: palo de 0,05 a 4 kg, media longitud de 0,3 a 3 m, retardo de 0 a 6 pasos."""
    r = np.random.default_rng(k)
    puntos = []
    for i in range(n_ep):
        ajustar_fisica(env, masspole=r.uniform(0.05, 4.0), length=r.uniform(0.3, 3.0))
        puntos.append(episodio(env, w, 7 * k + i, retardo=int(r.integers(0, 7))))
    ajustar_fisica(env)
    return float(np.mean(puntos))


w_ancha, _ = entropia_cruzada(env, np.random.default_rng(39), evaluador=evaluador_ancho)
print(f'aleatorización ancha: fuera, con 120 ms, {coma(rejilla_fisica(env, w_ancha, MASAS_X, LONG_X, retardo=6, n=4).mean(), 1)}; '
      f'en el mundo nominal, {coma(evaluar(env, w_ancha), 1)}; en la rejilla con 60 ms, {coma(rejilla_fisica(env, w_ancha, retardo=3).mean(), 1)}')

### Ejercicio 3 (en casa): un episodio o tres

Entrena con aleatorización midiendo cada candidato con **un solo episodio** (`n_ep=1`) y con tres, con cuatro semillas (5 a 8). Mide cada política en cinco mundos fijos que el entrenamiento no ha visto. ¿Gana siempre `n_ep=3`? ¿Coincide la puntuación con la que se entrena con la del test? ¿Qué harías antes de concluir nada?

In [ ]:
# Ejercicio 3
MUNDOS_TEST = [(0.1, 0.5, 0), (0.6, 1.0, 2), (1.0, 1.5, 3), (0.3, 0.8, 4), (0.8, 0.6, 1)]   # (kg, m, pasos de retardo)


def test(w, n=8):
    """Cinco mundos fijos, con semillas nuevas: el conjunto de test."""
    puntos = []
    for mp, L, d in MUNDOS_TEST:
        ajustar_fisica(env, masspole=mp, length=L)
        puntos.append(evaluar(env, w, n=n, semilla_base=77000, retardo=d))
    ajustar_fisica(env)
    return float(np.mean(puntos))


for semilla in (5, 6, 7, 8):
    for n_ep in (1, 3):
        ev = lambda w, k, n_ep=n_ep: evaluador_aleatorizado(env, w, k, n_ep=n_ep)
        w_x, h_x = entropia_cruzada(env, np.random.default_rng(semilla), evaluador=ev)
        print(f'semilla {semilla}, n_ep = {n_ep}: entrena con {coma(h_x[-1, 0], 1)}; test {coma(test(w_x), 1)}')

---

## Soluciones

**Ejercicio 1.** Con 48 pesos, ninguna de las 150 redes al azar pasa de 200 (la media baja a 18,0 y la mejor da 135,0); con 4 pesos, pasaba de 200 el 6 % de los 200 vectores al azar de la sección 2. La entropía cruzada gasta el triple, 1350 episodios, y llega a 300: su última vuelta da de media 268,6. Con esos mismos 1350 episodios al azar (450 redes), pasan de 200 solo dos, el 0,4 %; la mejor también llega a 300, pero es una de 450. La tarea es la misma; lo que crece es el espacio de pesos, y la fracción que funciona se vuelve despreciable. Por eso una red de millones de pesos no se busca al azar: hace falta usar cada medida para moverse, que es lo que hace el gradiente de política (la diapositiva 19).

**Ejercicio 2.** Con 120 ms, la nominal da 24,3 y la aleatorizada 32,1; con 160 ms, 24,3 y 27,4. Las dos se caen: fuera de lo variado, la aleatorizada no tiene por qué funcionar. La aleatorización no promete robustez en general: cambia el objetivo, de un mundo a una familia de mundos, y solo responde dentro de esa familia. Por eso se mide lo barato (la masa, la longitud, el retardo) para centrar los rangos donde está el robot de verdad, y se varía lo que no se puede medir. Y el mundo de fuera no era imposible: ensanchando la aleatorización hasta cubrirlo, la política da 300,0 fuera, 300,0 en el mundo nominal y 295,9 de media en la rejilla con 60 ms. Lo que faltaba era variarlo.

**Ejercicio 3.** Con la semilla 5, `n_ep=1` da 242,8 en el test y `n_ep=3`, 300,0; pero con la 7, al revés: 248,8 y 165,0. Con las cuatro semillas, las medias son 271,7 y 266,3: con cuatro semillas, la diferencia no se distingue de la suerte. Y la puntuación de entrenamiento no es la del test (con la semilla 5 y `n_ep=1`, 217,2 al entrenar y 242,8 en el test): la entropía cruzada elige por puntuación, y con pocos episodios elige también a los que tuvieron suerte. Antes de concluir, un conjunto de test aparte, varias semillas y la dispersión entre ellas: lo mismo que la diapositiva 31 pregunta a una tasa de éxito (en qué condiciones, con cuántos intentos y con su intervalo).

---

## Para llevarse de esta sesión

**Una política es una ley de control que se encuentra sin el modelo.** Cuatro pesos y el signo de w·x mantienen el palo de pie (300 de 300); con muchos pesos, al azar casi nada funciona y hace falta el gradiente: PPO, con su recorte.

**Un simulador con física da millones de intentos sin romper nada, pero se equivoca.** Con otros palos, la política nominal baja de 300 a 228,9 de media; con un retardo de 60 ms, a 147,4, y a 15,2 en el peor mundo.

**La aleatorización de dominio salva lo que se varió, y nada más.** Con 60 ms, 297,2 de media; fuera del rango, con 120 ms y palos de 2 a 4 kg, caen las dos.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*